# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 6 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'resume page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'Endpoints page', 'url': 'https://endpoints.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
baidu/Unlimited-OCR
Updated
2 days ago
•
429k
•
1.44k
empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF
Updated
2 days ago
•
971k
•
1.01k
zai-org/GLM-5.2
Updated
7 days ago
•
143k
•
3k
deepreinforce-ai/Ornith-1.0-35B-GGUF
Updated
5 days ago
•
157k
•
509
Qwen/Qwen-AgentWorld-35B-A3B
Updated
5 days a

In [23]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
Show the first version in english, then another version in Spanish.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nbaidu/Unlimited-OCR\nUpdated\n2 days ago\n•\n429k\n•\n1.44k\nempero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF\nUpdated\n2 days ago

In [19]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [20]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 16 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a pioneering AI company and the home of the machine learning community building the future. It offers a powerful platform where developers, researchers, and organizations come together to create, discover, and collaborate on machine learning models, datasets, and applications. Recognized as the collaboration platform for the ML community, Hugging Face fosters open innovation and enables unlimited hosting and sharing of public models and datasets.

---

## What They Offer

- **Models Hub**: Access and contribute to over 2 million AI models across various tasks and languages. The Hub features trending models such as OCR, large language models, and image editing tools.
  
- **Datasets**: Explore and share over 500,000 datasets to accelerate your AI development. These datasets are frequently updated and cover a broad spectrum of research and real-world applications.
  
- **Spaces**: Deploy and interact with AI applications directly on the platform. Examples include powerful image editors, OCR tools, video generation from prompts, and chatbots powered by state-of-the-art AI models.
  
- **Buckets & Storage**: Reliable storage solutions for datasets and models ensuring seamless collaboration and deployment.
  
- **Enterprise Solutions**: Tailored support for businesses through **Hugging Face PRO**, offering enhanced support, inference endpoints, providers, and managed services.

---

## Community & Collaboration

Hugging Face thrives as an open and inclusive community hub for AI and ML practitioners worldwide. Collaboration is supported through:

- **Open Source**: Hugging Face’s libraries and models are fully open-source and widely adopted.
- **Community Support**: Active forums, a Discord server, and frequent blog posts, research papers, and newsletters keep the community connected and inspired.
- **Learning Resources**: Tutorials, documentation, and daily papers to help users from beginners to experts grow their skills.
- **Developer Tools**: Integration with GitHub and APIs facilitate smooth workflows for developers.

---

## Company Culture

At Hugging Face, the culture revolves around openness, transparency, and innovation:

- **Collaborative Spirit**: The company emphasizes community-driven growth and open exchange of ideas.
- **Innovation Focus**: Committed to pushing the boundaries of AI by building accessible tools and platforms.
- **Diversity & Inclusion**: Actively fostering an inclusive environment welcoming talent from all backgrounds.
- **Growth & Learning**: Strong emphasis on continuous learning with access to cutting-edge research and a vibrant community.

---

## Customer Base

Hugging Face serves a vibrant ecosystem, including:

- Individual researchers and developers
- Academic institutions advancing AI knowledge
- Startups innovating with AI-driven products
- Large enterprises leveraging AI at scale with dedicated support plans and integrations

---

## Careers at Hugging Face

Join a forward-thinking team passionate about shaping the future of AI. Opportunities include roles in:

- Machine Learning Engineering
- Data Science and Research
- Software Development
- Community Management and Developer Relations
- Enterprise Solutions and Support

Hugging Face offers a dynamic environment to work alongside AI pioneers, contribute to open source projects, and impact the global AI community.

---

## Contact & Join

Discover the power of collaboration in AI at:  
**Website:** [huggingface.co](https://huggingface.co)  
**Community:** Discord, Forum, GitHub  
**Careers:** Explore open positions and join the AI revolution!

---

Experience innovation, community, and empowerment with Hugging Face —  
**The AI community building the future.**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [24]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [25]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Hugging Face Brochure  

---

## About Hugging Face  

Hugging Face is a vibrant AI community that is shaping the future of machine learning (ML). It serves as a collaborative platform where ML practitioners, researchers, and enterprises come together to build, share, and innovate on models, datasets, and AI applications. As the home of machine learning, Hugging Face provides an extensive repository of over 2 million models, 500,000+ datasets, and more than 1 million applications that users can explore, use, and build upon.  

---

## Platform & Solutions  

- **Models & Datasets:** Access a massive, diverse library of pretrained models and rich datasets continuously updated by a global community.  
- **Spaces:** Run and deploy AI apps directly on the platform, with featured projects showcasing cutting-edge AI applications such as image editing, OCR (Optical Character Recognition), video generation from images, and conversational AI.  
- **Enterprise Offering:** Specialized services including Hugging Face PRO, Enterprise Support, Inference Providers, and scalable Inference Endpoints geared towards corporate and large-scale AI deployments.  
- **Collaboration:** Hugging Face fosters open collaboration hosting unlimited public projects to accelerate AI innovation and democratize access to AI technology.  

---

## Community & Culture  

Hugging Face prides itself on being a vibrant, open, and inclusive community. It encourages collaboration through diverse channels such as Discord, forums, and GitHub. The company also supports learning and development through blogs, daily research papers, educational resources, and interactive discussions. Its culture embraces transparency, creativity, and community-driven progress — empowering developers, researchers, and organizations to contribute and evolve AI technology together.  

---

## Customers & Partners  

The platform serves a vast and active base spanning researchers, developers, AI enthusiasts, startups, and enterprises worldwide. By providing enterprise-grade AI infrastructure and support, it caters to organizations looking to integrate state-of-the-art machine learning models and inference capabilities into their products and services.  

---

## Careers at Hugging Face  

Hugging Face values innovation and impact. Careers here offer exciting opportunities to work with leading AI technologies, collaborate with top talent, and contribute to an open AI ecosystem. The company seeks passionate individuals excited about AI’s potential to reshape industries and communities globally.  

---

## Contact & Explore  

- **Website:** [huggingface.co](https://huggingface.co)  
- **Community:** Discord, Forum, GitHub  
- **Resources:** Blogs, Daily Papers, Docs  
- **Try:** Browse models, datasets, and AI apps for free  

---

# Tríptico de Hugging Face  

---

## Sobre Hugging Face  

Hugging Face es una comunidad dinámica de inteligencia artificial que está moldeando el futuro del aprendizaje automático. Sirve como una plataforma colaborativa donde practicantes, investigadores y empresas se unen para construir, compartir e innovar en modelos, conjuntos de datos y aplicaciones de IA. Como el hogar del aprendizaje automático, Hugging Face ofrece un enorme repositorio con más de 2 millones de modelos, más de 500 mil conjuntos de datos y más de 1 millón de aplicaciones para explorar y desarrollar.  

---

## Plataforma y Soluciones  

- **Modelos y Conjuntos de Datos:** Acceso a una vasta y diversa biblioteca de modelos preentrenados y conjuntos de datos, actualizados constantemente por una comunidad global.  
- **Spaces:** Ejecuta y despliega aplicaciones de IA directamente en la plataforma, con proyectos destacados como edición de imágenes, OCR (reconocimiento óptico de caracteres), generación de videos a partir de imágenes y IA conversacional.  
- **Ofertas para Empresas:** Servicios especializados incluyendo Hugging Face PRO, soporte empresarial, proveedores de inferencia y puntos de extremo para inferencia escalables dirigidos a implementaciones corporativas y a gran escala.  
- **Colaboración:** Hugging Face fomenta la colaboración abierta alojando proyectos públicos ilimitados para acelerar la innovación en IA y democratizar el acceso a esta tecnología.  

---

## Comunidad y Cultura  

Hugging Face se enorgullece de ser una comunidad abierta, inclusiva y dinámica. Promueve la colaboración a través de canales diversos como Discord, foros y GitHub. La empresa apoya el aprendizaje y desarrollo mediante blogs, artículos científicos diarios, recursos educativos y discusiones interactivas. Su cultura abraza la transparencia, la creatividad y el progreso impulsado por la comunidad, empoderando a desarrolladores, investigadores y organizaciones para avanzar juntos en la tecnología AI.  

---

## Clientes y Socios  

La plataforma atiende a una amplia y activa base que incluye investigadores, desarrolladores, entusiastas de la IA, startups y grandes empresas a nivel mundial. Ofreciendo infraestructura y soporte de IA a nivel empresarial, permite a organizaciones integrar modelos de aprendizaje automático de última generación y capacidades de inferencia en sus productos y servicios.  

---

## Trabaja con Nosotros  

Hugging Face valora la innovación y el impacto. Trabajar aquí significa contribuir con las tecnologías de IA líderes, colaborar con talento de primer nivel y participar en un ecosistema abierto de IA. Buscan personas apasionadas por el potencial de la IA para transformar industrias y comunidades en todo el mundo.  

---

## Contacto y Explora  

- **Sitio Web:** [huggingface.co](https://huggingface.co)  
- **Comunidad:** Discord, Foro, GitHub  
- **Recursos:** Blogs, Artículos diarios, Documentación  
- **Prueba:** Explora modelos, conjuntos de datos y aplicaciones de IA gratuitamente  

---

Join Hugging Face — where AI community builds the future!  
Únete a Hugging Face — donde la comunidad de IA construye el futuro!

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>